In [ ]:
# Gemma 4 Developer Agent — v10 (ONE-VARIABLE STRUCTURAL FIX: tool-call budget 22 -> 40; branched from v9)
# WHY v10 (evidence: the TRUSTWORTHY v9 SERIAL table, Gemma/Outputs/v9_2 L3805-3831, logged in
#   GEMMA_EXPERIMENT_LOG "v9 SERIAL VERDICT"): v9 scored 0/10 serial. The binding constraint on that run was
#   the 22-call TOOL BUDGET, NOT the 300s time wall:
#     - 8 of 10 tasks hit EXACTLY 22 calls (the cap).
#     - max duration was only 234s, well under the 300s cap -> nobody timed out.
#     - 3 tasks (rich_4079, rich_4076, fastapi_14246) emitted patch_chars=0 -> never reached an edit.
#   This is a budget-exhaustion signature. v9's prose (anti-thrash) tried to RECLAIM calls by asking nicely
#   and still pinned the cap. So v10 stops asking and RAISES the cap. This is the first STRUCTURAL
#   (config, not prose) lever and it targets the single measured failure mode directly.
#
# THE ONE VARIABLE: eval_config max_tool_calls 22 -> 40. Everything the MODEL sees (agent.yaml, sampling,
#   system.md prompt) is byte-identical to v9. No prompt changes. One knob.
#
# MEASUREMENT-CORRECTNESS CHANGE (NOT a second variable; changes nothing the model sees): PARALLEL = False.
#   v9's anchor-integrity guard PROVED concurrency corrupts verdicts (rich_4077 flipped True->False between
#   the parallel pass and the serial re-run; GEMMA_EXPERIMENT_LOG RETRACTION + "PARALLELIZATION IS REJECTED").
#   v10 runs SERIAL so its number is trustworthy from the first pass. The parallel/guard scaffold is removed
#   entirely — there is nothing to guard when we never run parallel.
#
# PRE-REGISTERED BAR (rule 2 — written BEFORE the run; held-out judge = the real 31B on the representative-10,
#   SERIAL):
#   - NULL it must beat: v9 serial = 0/10 on this exact sample.
#   - SUCCESS threshold: >=1/10 resolved AND at least one of the 3 budget-starved zero-patch tasks
#     (rich_4079, rich_4076, fastapi_14246) now emits a non-empty patch. (We want evidence the EXTRA CALLS
#     are what converts — i.e. the mechanism changed, not just a lucky flip.)
#   - MECHANISM CHECK (rule 13): read the tool_calls column. If raising the cap to 40 converts tasks, the
#     converters should be using >22 calls. If everything still dies at <=22 with 0 patches, the budget was
#     NOT the limiter and this hypothesis is FALSIFIED — report that as a null, do not spin it.
#   - CAVEAT (rule 9, stated not hidden): we still have NO v6-on-this-10 serial baseline, so 0/10 is
#     un-calibrated. v10 vs v9 is a clean within-agent delta (only the cap changed), which is interpretable
#     on its own; absolute standing vs v6 is NOT settled by this run.
#   DO NOT submit to the scored leaderboard on this result alone. v6/0.12 stays sacred live-best.

# ===== CELL 1: environment + wheelhouse =====
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# ROBUST wheelhouse discovery: the dataset mounts at DIFFERENT paths in interactive Draft Session
# (/kaggle/input/<slug>) vs batch/scoring (/kaggle/input/datasets/<owner>/<slug>). Search recursively
# for the actual .whl files instead of trusting one hardcoded path. Fall back gracefully if none found.
_WHEEL_HINTS = [
    '/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
    '/kaggle/input/gemma-4-developer-agent-wheelhouse',
]
_wheel_src = next((p for p in _WHEEL_HINTS if Path(p).is_dir()), None)
if _wheel_src is None:
    _found = glob.glob('/kaggle/input/**/*.whl', recursive=True)
    _wheel_src = str(Path(_found[0]).parent) if _found else None
print(f'[wheelhouse] source dir: {_wheel_src}')
for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
if _wheel_src:
    for w in Path(_wheel_src).glob('*.whl'):
        if 'cutlass' in w.name.lower(): continue
        target_name = (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
        target = tmp_whl / target_name
        if not target.exists():
            try: os.symlink(w, target)
            except FileExistsError: pass
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels...')
if wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
    importlib.invalidate_caches()
    print('Wheelhouse installation complete.')
else:
    print('[wheelhouse] WARNING: 0 wheels found. /kaggle/input contents:')
    for p in sorted(glob.glob('/kaggle/input/*')) + sorted(glob.glob('/kaggle/input/*/*')):
        print('   ', p)
    print('[wheelhouse] Skipping install. If swegemma/vllm import fails below, ADD the '
          'gemma-4-developer-agent-wheelhouse dataset as an input to this session.')


In [ ]:
# ===== CELL 2: write OUR agent directory inline (v10 = v9 agent BYTE-IDENTICAL; only eval cap differs) =====
from swegemma.models import load_tasks
def _resolve_input(candidates, sentinel):
    for c in candidates:
        if (Path(c) / sentinel).exists() or Path(c).is_dir() and sentinel == '':
            return Path(c)
    hits = glob.glob(f'/kaggle/input/**/{sentinel}', recursive=True) if sentinel else []
    if hits:
        return Path(hits[0]).parent
    raise FileNotFoundError(f'could not resolve input dir; tried {candidates} + glob for {sentinel!r}. '
                            f'/kaggle/input has: {sorted(glob.glob("/kaggle/input/*"))}')
DATA_DIR = _resolve_input([
    '/kaggle/input/competitions/gemma-4-developer-agent',   # batch/scoring
    '/kaggle/input/gemma-4-developer-agent',                 # interactive Draft Session
], 'tasks.jsonl')
print(f'[data] DATA_DIR = {DATA_DIR}')
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists(): shutil.rmtree(AGENT_DIR)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)

# v10 agent.yaml: IDENTICAL to v9 (sub-agent dropped; same tools).
(AGENT_DIR / 'agent.yaml').write_text('''name: sibling_conform_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
''', encoding='utf-8')

# v10 sampling: IDENTICAL to v9/v6 (temp 0.2, thinking 512).
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text('''temperature: 0.2
top_p: 0.95
max_output_tokens: 6144
thinking_config:
  thinking_budget: 512
  include_thoughts: true
''', encoding='utf-8')

# v10 eval_config: THE ONE VARIABLE. max_tool_calls 22 -> 40. Everything else held: max_time_minutes 5
# (so we can SEE whether extra calls or the time wall becomes the new limiter), max_turns bumped 40->60
# so turns don't become the artificial cap before 40 calls can be spent, timeout_seconds unchanged.
(AGENT_DIR / 'eval_config.yaml').write_text('''evaluation:
  max_time_minutes: 5
  max_tool_calls: 40
  max_turns: 60
  timeout_seconds: 120
''', encoding='utf-8')

# v10 system.md: BYTE-IDENTICAL to v9. The prompt still SAYS "~22 tool calls" on purpose would be a lie now;
# we update ONLY the single budget number it quotes to the agent so the stated budget matches the real cap.
# This is not a prose/strategy change — it is keeping the prompt's self-description truthful about the cap.
# (If we left "~22" the agent would self-limit to a budget it no longer has, confounding the experiment.)
(AGENT_DIR / 'prompts' / 'system.md').write_text('''You are an expert autonomous software engineer. You have a HARD budget of ~40 tool calls and 5 minutes. BE DECISIVE AND FAST.

## The one rule that matters: EDIT EARLY, DON'T EXPLORE.
You will run out of time if you keep searching. Spend AT MOST 3-4 tool calls locating the bug, then MAKE YOUR EDIT. One good minimal fix beats endless exploration. If unsure, make your best minimal change and submit - a plausible patch scores, endless searching scores nothing.

## Do NOT waste tool calls (this is what makes you run out before you fix the bug)
Your budget is small. Most failures are running out of calls while still searching. Obey:
- BATCH your searches. Need to find several symbols? Use ONE grep with alternation: `grep -nE 'isclass|currentframe|zlib|escape|_svg_hash' file.py` - not one grep per symbol. One call, all locations.
- NEVER repeat a search that already failed. If a grep returned nothing, a slightly reworded version of the SAME grep will also return nothing. Do not run it again. Change approach or move on.
- NEVER repeat a tool CALL that returned the SAME error. If a tool call fails (e.g. "mandatory parameter filepath not present"), the identical call will fail identically - do NOT retry it unchanged. Read the error, FIX the arguments it names, and call once more; if it fails a second time, switch tools (e.g. use run_command heredoc `cat > /path <<'EOF'` instead of write_file) or abandon that step. Looping the same failing call wastes your whole budget and times you out with a 0-char patch.
- After 2 searches that return nothing useful, STOP searching. Open the single most likely file with read_file and make your best edit from what you can see. Guessing an edit beats burning the budget to zero with a 0-char patch.
- Reading the same file region twice is a wasted call. Read once, remember it.
Every call you save on searching is a call you can spend making or correcting the fix.

## Strategy: conform to the pattern
Most bugs are a DEVIATION from a pattern the surrounding code already follows. Make the buggy code look like its correct siblings (the function before/after it, or similar functions in the same file). Fix the exact deviation (wrong constant, missing guard/branch, wrong exception, off-by-one, wrong attr/arg, missing return) - nothing else.

## Match the EXACT contract the problem describes (this is what the hidden tests check)
The hidden test asserts a PRECISE observable contract, not just "the behavior is roughly right". When your fix introduces or changes a user-facing string, error message, exception type, status code, or validation, you MUST match the exact wording and cover every obvious input variant:
- ERROR / VALIDATION messages: use the exact phrasing the problem statement implies, and mirror the style of the SIBLING check already in the file (same message shape, same field-name interpolation). Do NOT invent your own wording - a test often does `pytest.raises(..., match="exact phrase")`, so a paraphrase FAILS even when the logic is right.
- PHRASE THE MESSAGE IN THE PROBLEM'S OWN VOCABULARY, not a mechanical description of your check. The hidden test's expected message echoes the WORDS the problem uses to describe the bad state. If the problem says the data is "multi-line content", the message should assert it "must be a single line" (mirror the problem's own adjective/noun - here "multi-line" -> "single line") - do NOT write your own mechanical phrasing like "must not contain newlines". Scan the problem statement for the noun/adjective it uses for the invalid condition and build the message from THAT word, interpolating the field name the same way the sibling check does.
- INPUT VARIANTS: if you reject newlines, reject "\\n" AND "\\r" AND "\\r\\n". If you guard one special char, guard the whole obvious class. Tests parametrize across all of them.
- Apply the change to EVERY field/argument/branch the problem mentions, not just the first - if the problem says "id or event", fix BOTH, mirroring how an existing validated field is wired.
- A self-written test only counts as verification if it asserts the SAME exact string and the SAME variants the problem implies. If your test passes but only checks your own paraphrase, it proves nothing. Prefer asserting the behavior described in the problem's own words.

## Workflow (tight)
1. Read the problem statement. Extract the file/function/error AND the exact contract (messages, variants, every field named). Open the code with read_file (or ONE batched `grep -nE` via run_command if no path given). MAX 3-4 locate calls.
2. Make the minimal fix with edit_file (small edits only). Match existing conventions/strings/types EXACTLY, including the sibling's message style.
3. Call submit_patch ONCE. The moment it returns status "ok", you are DONE: write a one-sentence summary and make NO more tool calls. NEVER call submit_patch twice.

## Hard rules
- NEVER modify/create/delete test files (test_*.py, *_test.py, tests/) or runner config (conftest.py, pytest.ini, pyproject.toml) - the harness resets them; it pollutes your diff or auto-fails.
- write_file is JAILED to /workspace and rejects /tmp. For scratch, use run_command heredoc: `cat > /tmp/x.py <<'EOF'`. For a NEW repo file, use run_command heredoc to the real path. Use edit_file ONLY for existing files.
- Offline: no pip install. Do NOT refactor unrelated code. Do NOT run bare pytest (it times out) - if you test, run ONLY the one target test.
- Do NOT conclude without a non-empty patch.
''', encoding='utf-8')

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
print(f'Agent dir written (v10 = v9 agent, max_tool_calls 22->40): {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks')


In [ ]:
# ===== CELL 3: start vLLM server =====
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
def _resolve_model():
    cands = [
        '/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2',
        '/kaggle/input/models/google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2',
    ]
    for c in cands:
        if Path(c, 'config.json').exists():
            return Path(c)
    hits = glob.glob('/kaggle/input/**/gemma-4-31b-it-qat-w4a16-ct/**/config.json', recursive=True)
    if hits:
        return Path(hits[0]).parent
    raise FileNotFoundError(f'w4a16 model not found; /kaggle/input: {sorted(glob.glob("/kaggle/input/*"))}')
MODEL_PATH = _resolve_model()
print(f'[model] MODEL_PATH = {MODEL_PATH}')
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
    enable_lora=True, max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp_size, startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')


In [ ]:
# ===== CELL 4: A/B RUN — representative 10 tasks; SERIAL ONLY (parallel REJECTED, see header) =====
import asyncio, concurrent.futures, yaml
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
# quiet the known-harmless log spam (ensurepip-in-sandbox warnings; litellm orphaned-worker asyncio noise).
# These are NOT failures (GEMMA_EXPERIMENT_LOG "harmless log noise"); silencing them makes the serial
# per-task verdicts readable instead of buried under hundreds of lines.
import logging
for _n in ('swegemma.sandbox.subprocess', 'asyncio', 'LiteLLM', 'litellm'):
    logging.getLogger(_n).setLevel(logging.CRITICAL)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())
raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
ev = raw.get('evaluation', raw)
timeout_seconds = int(ev.get('timeout_seconds', 300)); max_tool_calls = int(ev.get('max_tool_calls', 100))
max_time_minutes = float(ev.get('max_time_minutes', 60.0))
turns_raw = ev.get('max_turns', ev.get('max_llm_calls')); max_turns = int(turns_raw) if turns_raw is not None else None
# SAME representative sample as v9 (so v10 vs v9 is a clean within-sample within-agent delta: only the
# tool-call cap changed). 10 SMALL_SINGLE_FILE, repo-stratified rich4/fastapi4/requests2, seed=42.
REPR_SAMPLE_IDS = [
    "rich_4077", "rich_4079", "rich_4076", "rich_3469",
    "fastapi_14485", "fastapi_14487", "fastapi_14246", "fastapi_14356",
    "requests_7315", "requests_7502",
]
ANCHOR_IDS = ["rich_4077", "rich_4079", "rich_4076"]
by_iid = {t.instance_id: t for t in tasks}
SAMPLE_TASKS = [by_iid[i] for i in REPR_SAMPLE_IDS if i in by_iid]
missing = [i for i in REPR_SAMPLE_IDS if i not in by_iid]
print(f'Representative sample: {len(SAMPLE_TASKS)}/{len(REPR_SAMPLE_IDS)} tasks (missing: {missing})')
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
    timeout_seconds=timeout_seconds, max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls, max_turns=max_turns,
    limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
        token_threshold=14336, event_retention_size=5),
    graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels', verbose=False,
)
evaluator = Evaluator(eval_config)

def score_one(task, idx, total):
    r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=total)
    return {'id': task.instance_id, 'repo': task.repo.split('/')[-1],
            'resolved': bool(r.resolved), 'duration_s': round(r.duration_seconds, 1),
            'tool_calls': r.tool_calls, 'patch_chars': len(r.agent_patch or '')}

# SERIAL ONLY. No parallel path, no anchor guard — concurrency was proven to corrupt verdicts (v9
# GEMMA_EXPERIMENT_LOG RETRACTION), so there is nothing to guard. One task at a time against the shared
# vLLM server = the only trustworthy mode on this harness.
import time as _time
t_start = _time.time()
rows = []
for i, t in enumerate(SAMPLE_TASKS):
    print(f'[serial {i+1}/{len(SAMPLE_TASKS)}] {t.instance_id} ...', flush=True)
    row = score_one(t, i + 1, len(SAMPLE_TASKS))
    rows.append(row)
    print(f'  [done] {t.instance_id}: resolved={row["resolved"]} '
          f'dur={row["duration_s"]}s calls={row["tool_calls"]} chars={row["patch_chars"]}', flush=True)
wall = _time.time() - t_start
print(f'[serial] wall time {wall:.0f}s for {len(SAMPLE_TASKS)} tasks', flush=True)

tdf = pd.DataFrame(rows)
print(f'\n===== v10 SUMMARY  [SERIAL]  (REPRESENTATIVE 10, SMALL_SINGLE_FILE) =====')
print('CHANGE vs v9: max_tool_calls 22 -> 40 (ONE variable). Bar: beat v9=0/10 AND >=1 of the 3 zero-patch')
print('budget-starved tasks (rich_4079, rich_4076, fastapi_14246) emits a non-empty patch.')
print(tdf.to_string(index=False))
print(f'\nRESOLVED: {tdf.resolved.sum()}/{len(tdf)}  ({100*tdf.resolved.mean():.0f}%)')
print('by repo:', tdf.groupby('repo')['resolved'].agg(['sum', 'count']).to_dict('index'))
for iid in ANCHOR_IDS:
    row = tdf[tdf.id == iid]
    if len(row): print(f'  anchor {iid}: resolved={bool(row.resolved.iloc[0])}')
# MECHANISM CHECK (rule 13): did the extra budget actually get USED, and did the zero-patch tasks recover?
print(f'Max tool_calls used: {int(tdf.tool_calls.max())} (cap now 40) | max dur: {round(tdf.duration_s.max())}s (cap 300)')
_prev_zero = ['rich_4079', 'rich_4076', 'fastapi_14246']  # emitted 0 patch chars under v9
for iid in _prev_zero:
    row = tdf[tdf.id == iid]
    if len(row):
        print(f'  [mechanism] {iid}: now calls={int(row.tool_calls.iloc[0])} '
              f'patch_chars={int(row.patch_chars.iloc[0])} resolved={bool(row.resolved.iloc[0])}')
_over22 = tdf[tdf.tool_calls > 22]
print(f'  [mechanism] tasks that used MORE than 22 calls (only possible with the raised cap): '
      f'{list(_over22.id)} ({len(_over22)}/10)')


In [ ]:
# ===== CELL 5: package submission.zip =====
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES
zip_path = Path(shutil.make_archive(str(WORKING_DIR / 'submission'), 'zip', root_dir=AGENT_DIR))
with zipfile.ZipFile(zip_path, 'r') as zf:
    total = sum(i.file_size for i in zf.infolist())
    assert total <= MAX_SUBMISSION_SIZE_BYTES, f'too big: {total}'
    for info in zf.infolist():
        if not info.is_dir():
            assert Path(info.filename).suffix.lower() in ALLOWED_SUBMISSION_EXTENSIONS, info.filename
print(f'Created {zip_path} ({zip_path.stat().st_size/1024:.1f} KiB)')
